# 01 · Synthetic rider audio (scripts → TTS → real + synthetic noise & phone channel)

**Settings:** Accelerator **GPU T4** · Internet **on** · Secret **HF_TOKEN** (Add-ons → Secrets).
Indic Parler-TTS is gated with automatic approval: open huggingface.co/ai4bharat/indic-parler-tts once and accept.

Inputs: the repo folder as a dataset (or set `REPO_URL`). **Round 2:** also attach the previous run of this notebook —
its TTS renders are reused (same script, same voice, same seed), so only the new scripts are synthesised.
Optional: your own road recordings as a dataset with `traffic/ horn/ engine/ wind/ rain/ babble/ kitchen/` folders.

Run all. Resumable. Output: `/kaggle/working/rv-audio` → **Save Version** → attach to notebook 02.

In [ ]:
# MUST run before anything imports transformers: parler-tts pins protobuf 4.x, which breaks the TensorFlow that
# transformers would otherwise import on Kaggle images.
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
# ---- settings
REPO_URL = ""
RAW_SCRIPTS = ["data/scripts/m1_wedge.raw.jsonl", "data/scripts/m1_round2.raw.jsonl"]
FREEZE_SPLITS = "data/scripts/m1_wedge.jsonl"   # round-1 scripts keep their split -> results stay comparable
FETCH_MS_SNSD = True                            # real noise (CC0 / CC BY-SA 3.0) into the chain
MY_NOISE = ""                                   # e.g. "/kaggle/input/rv-noise" - your own recordings, merged in
OUT = "/kaggle/working/rv-audio"
TIERS = "clean,20,10,5,0"
MAX_TTS_MINUTES = 600

In [ ]:
import os, subprocess, sys, glob, shutil, json
from pathlib import Path

def run(cmd, cwd=None, env=None, check=True):
    """Run a shell command and stream its output live (every stage is a subprocess, so GPU memory is freed after it)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "it/s]" not in line:
            print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")
    return p.returncode

# --- scratch outside /kaggle/working, so code, model and features are not saved into the notebook output
TMP = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")
# --- code: a Kaggle dataset upload of the repo folder (newest spec version wins), or a git URL
CODE = TMP / "rvd"
if not CODE.exists():
    found = [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/spec/decision_set.json", recursive=True)]
    found.sort(key=lambda d: json.loads((d / "spec" / "decision_set.json").read_text()).get("version", "0"))
    if found:
        shutil.copytree(found[-1], CODE)
    elif REPO_URL:
        run(f"git clone --depth 1 {REPO_URL} {CODE}")
    else:
        raise SystemExit("Add the repo folder as a Kaggle dataset (Add Input -> Datasets), or set REPO_URL")
print("code:", CODE, "| spec", json.loads((CODE / "spec" / "decision_set.json").read_text())["version"])

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as e:  # harmless for public models; needed for the gated TTS model in notebook 01
    print("HF_TOKEN secret not available:", type(e).__name__, "- fine for public models")

In [ ]:
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
run("pip install -q git+https://github.com/huggingface/parler-tts.git soundfile")   # pins transformers 4.46.1 - fine here
run("ffmpeg -hide_banner -encoders 2>/dev/null | grep -E 'libopus|amrnb' || echo 'no opus/amr encoder: G.711 fallback'", check=False)

In [ ]:
# ---- 1. scripts: lint + dedup + gold labels (rules) + script-level split (frozen for round-1 scripts)
Path(OUT).mkdir(parents=True, exist_ok=True)
raw = " ".join(str(CODE / r) for r in RAW_SCRIPTS)
run(f"python src/scripts.py build --raw {raw} --out {OUT}/scripts.jsonl --freeze-splits {CODE / FREEZE_SPLITS}", cwd=CODE)

In [ ]:
# ---- 2. reuse earlier renders (attach the previous run of this notebook as an input)
prev = [Path(p).parent for p in glob.glob("/kaggle/input/**/tts/tts_manifest.jsonl", recursive=True)]
tts_dir = Path(OUT) / "tts"
if prev and not (tts_dir / "tts_manifest.jsonl").exists():
    shutil.copytree(prev[0], tts_dir)
    print("reusing", sum(1 for _ in open(tts_dir / "tts_manifest.jsonl")), "renders from", prev[0])

In [ ]:
# ---- 3. TTS (resumable; progress printed every 10 renders)
run(f"python src/tts.py --scripts {OUT}/scripts.jsonl --out {OUT}/tts --max-minutes {MAX_TTS_MINUTES}", cwd=CODE)

In [ ]:
# ---- 4. real noise: MS-SNSD noise folders (+ your own recordings)
NOISE = TMP / "noise"
if FETCH_MS_SNSD:
    run(f"python src/fetch_noise.py --out {NOISE}", cwd=CODE)
if MY_NOISE:
    for cat_dir in Path(MY_NOISE).iterdir():
        if cat_dir.is_dir():
            dst = NOISE / cat_dir.name / "train"; dst.mkdir(parents=True, exist_ok=True)
            for i, f in enumerate(sorted(cat_dir.rglob("*.wav"))):  # every 5th own recording -> test noise
                shutil.copy(f, (NOISE / cat_dir.name / ("test" if i % 5 == 0 else "train") / f"own_{f.name}"))

In [ ]:
# ---- 5. noise + channel chain -> clips + manifest (CPU)
nd = f"--noise-dir {NOISE}" if NOISE.exists() else ""
run(f"python src/augment.py --tts-manifest {OUT}/tts/tts_manifest.jsonl --scripts {OUT}/scripts.jsonl --out {OUT}/clips "
    f"--tiers {TIERS} {nd}", cwd=CODE)

In [ ]:
# ---- 6. check: counts + listen to a few clips
import collections, random
from IPython.display import Audio, display
man = [json.loads(l) for l in open(f"{OUT}/clips/manifest.jsonl")]
print(collections.Counter((m["split"], m["tier"]) for m in man))
print("noise sources:", collections.Counter(s.split(":")[1] for m in man for s in m["aug"].get("noise_src", [])))
flag = [json.loads(l) for l in open(f"{OUT}/tts/tts_manifest.jsonl")]
print("renders with odd length:", sum(not f["length_ok"] for f in flag), "/", len(flag))
for m in random.Random(0).sample(man, 4):
    print(m["clip_id"], m["situation"], m["tier"], m["aug"].get("noise_src"), m["aug"].get("codec_used"))
    display(Audio(f"{OUT}/clips/{m['path']}"))